# Notebook 02 — Game Event Extraction

This notebook parses a saved gol.gg game HTML page and extracts
events from the "Gold graph & Timeline" section.

For each event, we will extract:

- Game ID
- Event type
- Team
- Side
- Event timestamp
- Event timestamp in seconds
- Icon source URL

In [ ]:
import re #for regular expressions for timestamps
from pathlib import Path #working with file paths

import pandas as pd #for data creation and final table
from bs4 import BeautifulSoup #parses HTMLs



In [8]:
# Setting up example code with one specific game

GAME_ID = "83713"

html_path = Path("../../data/raw/html") / f"game_{GAME_ID}.html"

print("HTML path:", html_path.resolve())
print("File exists:", html_path.exists())

HTML path: C:\Users\boyle\League_Of_Legends_Herald_Analysis\data\raw\html\game_83713.html
File exists: True


In [ ]:
# Load the HTML

with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

print("HTML characters:", len(html))

HTML characters: 625498


In [10]:
# Parse the HTML with Beautiful Soup

soup = BeautifulSoup(html, "lxml")

print("HTML parsed successfully.")

HTML parsed successfully.


In [12]:
#find timeline container
# From the HTML we expect to find a container that looks like this:
# <div class="col-12 flex-wrap">
# Inside that div we will find something like this:

#<span class="empty_action title_blue">GEN</span>
#<span class="empty_action title_red">IG</span>


#Finding the container

timeline_container = None

for div in soup.find_all("div"):
    classes = div.get("class", [])

    if (
        "col-12" in classes
        and "flex-wrap" in classes
        and div.find(class_="title_blue")
        and div.find(class_="title_red")
    ):
        timeline_container = div
        break

print("Timeline container found:", timeline_container is not None)

Timeline container found: True


In [13]:
# Extract Team Names

blue_team = timeline_container.find(class_="title_blue").get_text(strip=True)
red_team = timeline_container.find(class_="title_red").get_text(strip=True)

print("Blue team:", blue_team)
print("Red team:", red_team)

Blue team: GEN
Red team: IG


In [14]:
#Create Team Mapping

team_names = {
    "blue_action": blue_team,
    "red_action": red_team
}

team_names

{'blue_action': 'GEN', 'red_action': 'IG'}

In [16]:
#Inspect Timeline Structure

timeline_spans = timeline_container.find_all("span", recursive=False)

print("Number of direct span elements:", len(timeline_spans))

for i, span in enumerate(timeline_spans[:5]):
    print(f"\n--- Span {i} ---")
    print(span.prettify()[:1000])

Number of direct span elements: 10

--- Span 0 ---
<span style="display:inline-block">
 <span class="empty_action title_blue">
  GEN
  <br/>
  <br/>
 </span>
 <span class="empty_action title_red">
  IG
  <br/>
  <br/>
 </span>
</span>


--- Span 1 ---
<span style="display:inline-block">
 <span class="blue_action">
  <img alt="First blood" class="champion_icon_light" src="../_img/firstblood3.png"/>
  <br/>
  3:45
 </span>
 <span class="empty_action">
  <br/>
 </span>
</span>


--- Span 2 ---
<span style="display:inline-block">
 <span class="empty_action">
  <br/>
 </span>
 <span class="red_action">
  5:43
  <br/>
  <img alt="Chemtech Drake" class="champion_icon_light" src="../_img/chemtech-dragon.png"/>
 </span>
</span>


--- Span 3 ---
<span style="display:inline-block">
 <span class="blue_action">
  <img alt="Mountain Drake" class="champion_icon_light" src="../_img/mountain-dragon.png"/>
  <br/>
  11:20
 </span>
 <span class="empty_action">
  <br/>
 </span>
</span>


--- Span 4 ---
<s

In [17]:
#identify event wrappers

event_wrappers = timeline_spans[1:]

print("Number of event wrappers:", len(event_wrappers))

Number of event wrappers: 9


In [18]:
# create an empty list for our events

events = []

In [19]:
#extraction loop

for wrapper in event_wrappers:

    # Look at the direct child lane spans inside this event wrapper
    lanes = wrapper.find_all("span", recursive=False)

    for lane in lanes:

        # Get the CSS classes for this lane
        classes = lane.get("class", [])

        # Determine whether this is blue or red side
        side_class = next(
            (side for side in team_names if side in classes),
            None
        )

        # Ignore spans that aren't event lanes
        if side_class is None:
            continue

        # Find the event icon
        img = lane.find("img")

        # Ignore empty lanes
        if img is None:
            continue

        # Extract the timestamp
        timestamp_match = re.search(
            r"(?<!\d)(\d{1,2}:\d{2})(?!\d)",
            lane.get_text(" ", strip=True)
        )

        if timestamp_match is None:
            continue

        timestamp = timestamp_match.group(1)

        # Convert CSS class into human-readable side/team
        side = "blue" if side_class == "blue_action" else "red"
        team = team_names[side_class]

        # Extract event type from image alt text
        event_type = img.get("alt")

        # Save the event
        events.append({
            "game_id": GAME_ID,
            "event_type": event_type,
            "team": team,
            "side": side,
            "event_time": timestamp,
            "icon_src": img.get("src")
        })

In [20]:
#see the events

events

[{'game_id': '83713',
  'event_type': 'First blood',
  'team': 'GEN',
  'side': 'blue',
  'event_time': '3:45',
  'icon_src': '../_img/firstblood3.png'},
 {'game_id': '83713',
  'event_type': 'Chemtech Drake',
  'team': 'IG',
  'side': 'red',
  'event_time': '5:43',
  'icon_src': '../_img/chemtech-dragon.png'},
 {'game_id': '83713',
  'event_type': 'Mountain Drake',
  'team': 'GEN',
  'side': 'blue',
  'event_time': '11:20',
  'icon_src': '../_img/mountain-dragon.png'},
 {'game_id': '83713',
  'event_type': 'First Tower',
  'team': 'IG',
  'side': 'red',
  'event_time': '14:30',
  'icon_src': '../_img/firsttower2.png'},
 {'game_id': '83713',
  'event_type': 'Rift Herald',
  'team': 'GEN',
  'side': 'blue',
  'event_time': '15:27',
  'icon_src': '../_img/herald-icon.png'},
 {'game_id': '83713',
  'event_type': 'Hextech Drake',
  'team': 'GEN',
  'side': 'blue',
  'event_time': '16:47',
  'icon_src': '../_img/hextech-dragon.png'},
 {'game_id': '83713',
  'event_type': 'Hextech Drake',
  

In [21]:
#create a dataframe

events_df = pd.DataFrame(events)

events_df

,game_id,event_type,team,side,event_time,icon_src
0,83713,First blood,GEN,blue,3:45,../_img/firstblood3.png
1,83713,Chemtech Drake,IG,red,5:43,../_img/chemtech-dragon.png
2,83713,Mountain Drake,GEN,blue,11:20,../_img/mountain-dragon.png
3,83713,First Tower,IG,red,14:30,../_img/firsttower2.png
4,83713,Rift Herald,GEN,blue,15:27,../_img/herald-icon.png
5,83713,Hextech Drake,GEN,blue,16:47,../_img/hextech-dragon.png
6,83713,Hextech Drake,GEN,blue,22:38,../_img/hextech-dragon.png
7,83713,Hextech Drake,GEN,blue,28:30,../_img/hextech-dragon.png
8,83713,Nashor,GEN,blue,30:10,../_img/nashor-icon.png


In [ ]:
#add event numbers for organization (we can use this later for looking at sequence importance for winning)

events_df.insert(
    1,
    "event_number",
    range(1, len(events_df) + 1)
)

events_df

,game_id,event_number,event_type,team,side,event_time,icon_src
0,83713,1,First blood,GEN,blue,3:45,../_img/firstblood3.png
1,83713,2,Chemtech Drake,IG,red,5:43,../_img/chemtech-dragon.png
2,83713,3,Mountain Drake,GEN,blue,11:20,../_img/mountain-dragon.png
3,83713,4,First Tower,IG,red,14:30,../_img/firsttower2.png
4,83713,5,Rift Herald,GEN,blue,15:27,../_img/herald-icon.png
5,83713,6,Hextech Drake,GEN,blue,16:47,../_img/hextech-dragon.png
6,83713,7,Hextech Drake,GEN,blue,22:38,../_img/hextech-dragon.png
7,83713,8,Hextech Drake,GEN,blue,28:30,../_img/hextech-dragon.png
8,83713,9,Nashor,GEN,blue,30:10,../_img/nashor-icon.png
